# V8 five sentence folds
Same CNN, raw TRT_total, learning rate 0.0001, seeds 45/62, real and arbitrary labels. All 12 readers remain separate examples.

Each fold fits 66 sentences, validates on 22 different sentences, tests on 22 other sentences. Every sentence is tested once. Fold 0 exactly preserves the old V8 test/validation/fit split: reuse its four verified result archives. The other four folds need **16 new fits**. No saved model weights. Results expected to use roughly 1–5 MB; no packages or datasets installed/downloaded by this notebook.

This is an exploratory extension. A fivefold mean and a 110-sentence out-of-fold macro-F1 are both reported; they are different aggregations.

## 1. Setup
Select a GPU runtime. Mount the account that has space and access to the shared Thesis folder. Existing input/result folders stay unchanged.

In [ ]:
import os, sys, subprocess, json
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
import tensorflow as tf
assert tf.config.list_physical_devices('GPU'), 'Select Runtime > Change runtime type > GPU, then reconnect and rerun setup.'
from google.colab import drive
drive.mount('/content/drive')
repo = '/content/teco-v8-sentence-cv-code'
run_ref = 'v8-sentence-cv-v1'
if not os.path.isdir(repo):
    subprocess.run(['git','clone','-q','--branch',run_ref,'--single-branch',
                    'https://github.com/parmisbathayan/reproduce-teco-results.git',repo],check=True)
os.chdir(repo)
assert subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip() == subprocess.check_output(['git','rev-parse',run_ref+'^{commit}'],text=True).strip(), 'Wrong source revision.'
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(), 'Use a clean fixed checkout.'
base = '/content/drive/MyDrive/Thesis/Results/reproduce_yaser_codes/eegonly'
local_dir = '/content/teco-v8-sentence-cv-results'
mirror_dir = base + '/v8_sentence_cv_5fold_45_62'
command = [sys.executable,'-u','-B','src/v8_sentence_cv.py',
    '--trt-dir','/content/drive/MyDrive/Thesis/Data/TRT_Total',
    '--labels-csv','/content/drive/MyDrive/Thesis/Data/teco_sentiment_labels_task1.csv',
    '--anchor-dir',base+'/v8_light_seen_readers_45_62',
    '--reference-dir',base+'/v4_low_lr_confirmation_45_62',
    '--local-dir',local_dir,'--mirror-dir',mirror_dir]
def stream_run(cmd):
    process = subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in process.stdout:
            print(line,end='',flush=True)
        rc = process.wait()
    except BaseException:
        process.terminate()
        process.wait()
        raise
    if rc:
        raise subprocess.CalledProcessError(rc,cmd)
print('Source:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
print('New Drive results folder:',mirror_dir)

## 2. Check the splits and reuse old jobs
This cell does **not train**. It reconstructs the original V8 plan, checks unchanged data/model/runtime, checks all five balanced folds, verifies and imports the four old jobs, and scores the saved V4 predictions on the same test groups. It should report **4/20 verified**, leaving **16 new fits** on the first run. On resume it can report more.

If it stops, send the error. Do not delete or overwrite the old results to bypass a failed check.

In [ ]:
stream_run(command + ['--prepare-only'])
import pandas as pd
from IPython.display import display
print('V4 on the five matched sentence groups (group 0 is the old V8 test set):')
display(pd.read_csv(os.path.join(local_dir,'v4_matched_fold_summary.csv')))

## 3. Run or resume the remaining fits
Run this cell after section 2 passes. Each saved fit is verified from its predictions and mirrored to the new Drive folder. Completed verified jobs are skipped. GPU training settings are unchanged; the code reports a heartbeat while a fit is running.

The original four V8 jobs are reused, not retrained. Missing/corrupt source archives cause a stop rather than silently training replacements.

In [ ]:
stream_run(command)

## 4. Verify and read the results
This cell verifies all 20 jobs without training. `summary_fold_mean.csv` averages reader F1 over folds and seeds. `summary_oof_110.csv` first combines each reader/seed's predictions across all 110 sentences and then averages their F1. Use the latter alongside V4's original 110-sentence aggregation; the protocols still differ in reader familiarity, fitting size and sentence overlap.

One arbitrary mapping, two initializations and one fivefold partition do not provide a formal permutation p-value.

In [ ]:
stream_run(command + ['--verify-only'])
print('Mean over sentence folds, readers and seeds:')
display(pd.read_csv(os.path.join(local_dir,'summary_fold_mean.csv')))
print('110-sentence out-of-fold scores, averaged over readers and seeds:')
display(pd.read_csv(os.path.join(local_dir,'summary_oof_110.csv')))
print('V4 on the matching sentence groups:')
display(pd.read_csv(os.path.join(local_dir,'v4_matched_fold_summary.csv')))
import shutil
backup = shutil.make_archive('/content/v8_sentence_cv_results','zip',local_dir)
print('Verified cloud-runtime backup:',backup,os.path.getsize(backup),'bytes')
print('Drive mirror:',mirror_dir)
print('Before ending the runtime, confirm the new result files are visible on the Drive website. Mounted readback alone does not prove server sync.')